# Lab R.1 &mdash; Hybrid retrieval

**About 25 minutes** &middot; Day 2 &middot; RAG, vector stores &amp; agent memory

In Module 3, AskOps searched runbooks by the words in their titles. Here you search the full runbook text in two ways, by meaning and by exact words, and merge the two lists.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The shared helpers are in `rag_kit.py`, next to this notebook.

**The result:** a hit-rate table that shows hybrid search beating each search on its own, and a model answer built from the chunks hybrid search found.

## Step 1 &mdash; Chunks and metadata

`rag_kit` splits each runbook in `data/runbooks/` at its `##` headings, so one chunk is one section:
*Symptoms*, *Checks*, *Fix* or *Escalation*. Each chunk starts with the runbook id and title, and
carries **metadata**: labels stored next to the text, which you can filter on.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")                    # the model libraries print a lot on first import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import rag_kit as kit

chunks = kit.all_chunks()
print(len(chunks), "chunks from", len(kit.load_runbooks()), "runbooks\n")
print(chunks[0]["text"], "\n")
print(chunks[0]["metadata"])

**You should see:** 52 chunks from 13 runbooks, then the *Symptoms* chunk of RB-101 and its metadata: `service`, `access`, `updated` and more.

## Step 2 &mdash; Vector search, with a metadata filter

`build_collection()` embeds every chunk and stores it in Chroma, in memory. A `where` filter limits
what a search may return. RB-111 is marked `access: restricted`, so an on-call engineer searches with
`{"access": "ops"}`. This is how retrieval respects access rights: the model never sees what it may
not show.

In [ ]:
col = kit.build_collection(chunks)
q = "We need to run SQL directly on the production database during an incident"
print("no filter   :", kit.vector_search(col, q, k=3))
print("ops only    :", kit.vector_search(col, q, k=3, where={"access": "ops"}))
print("ledger only :", kit.vector_search(col, "service is down", k=3, where={"service": "ledger"}))

**You should see:** with no filter, RB-111 comes first. With the ops filter it is gone. The ledger filter returns only RB-107 chunks.

## Step 3 &mdash; Two searches that disagree

`kit.BM25` is **keyword search**, the formula Elasticsearch uses. It likes rare, exact words. Vector
search compares **meaning**, so it does not need the same words. RB-113 is *Transaction service runs
out of memory*.

In [ ]:
bm25 = kit.BM25(chunks)
for question in ("OOMKilled at month end", "app killed because it used too much RAM"):
    print(question)
    print("  vector :", kit.vector_search(col, question, k=3))
    print("  keyword:", bm25.search(question, k=3))

**You should see:** for *OOMKilled*, keyword search puts RB-113 first, because `OOMKilled` is a rare
exact word, and vector search does not. For *too much RAM* it is the other way round: no runbook
says "RAM", but vector search knows RAM means memory. Each search is right where the other is wrong.

## Step 4 &mdash; Merge the two lists with RRF

The two scores are on different scales, so you cannot add them. **Reciprocal rank fusion (RRF)** uses
only the rank: a chunk at rank *r* gets `1 / (60 + r)` points from each list. A chunk near the top of
**both** lists wins.

In [ ]:
def rrf(ranked_lists, c=60):
    """Merge ranked lists of chunk ids into one list, best first."""
    scores = {}
    for ranked in ranked_lists:
        for rank, chunk_id in enumerate(ranked, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (c + rank)
    return sorted(scores, key=lambda cid: -scores[cid])

def hybrid_search(question, k=5):
    """Take 20 from each search, merge them, keep the best k."""
    return rrf([kit.vector_search(col, question, k=20), bm25.search(question, k=20)])[:k]

for question in ("OOMKilled at month end", "app killed because it used too much RAM"):
    print(f"{question:42} hybrid: {hybrid_search(question, k=3)}")

**You should see:** RB-113 in the top 2 for **both** questions.

## Step 5 &mdash; Measure it on 14 questions

`data/questions.json` has 14 on-call questions. For each one it names the chunk that holds the
answer. **Hit rate at 3** counts how often that chunk is in the top 3.

In [ ]:
questions = kit.load_questions()

def hits_at_3(search):
    return sum(q["chunk"] in search(q["question"])[:3] for q in questions)

for name, search in [("vector",  lambda q: kit.vector_search(col, q, k=3)),
                     ("keyword", lambda q: bm25.search(q, k=3)),
                     ("hybrid",  lambda q: hybrid_search(q, k=3))]:
    n = hits_at_3(search)
    print(f"{name:8} {n:2}/{len(questions)}  " + "#" * n)

**You should see:** hybrid scores higher than vector and keyword alone. It still misses some questions. Lab R.2 works on those.

## The result &mdash; an answer from the chunks

This is the **generation** half of RAG. The top 3 hybrid chunks go into a prompt that tells the
sandbox model to answer only from them, and to cite the runbook.

In [ ]:
question = "The transaction service keeps getting OOMKilled at month end. What do I do?"
top = hybrid_search(question, k=3)
text_of = {c["id"]: c["text"] for c in chunks}
passages = "\n\n".join(text_of[cid] for cid in top)
reply, tokens = kit.chat("Answer the on-call engineer using ONLY these runbook passages. "
                         "Cite the runbook id. At most 3 sentences.\n\n"
                         f"Passages:\n{passages}\n\nQuestion: {question}", max_tokens=200)
print("retrieved:", top)
print("\n" + reply)
print(f"\n[{tokens} tokens]")

**You should see:** RB-113 chunks retrieved, and an answer built from them that cites RB-113. The
model did not need to know anything about your runbooks. Retrieval gave it the text.